In [ ]:
# Outside of the sources mentioned in README, we use data from the
# BISA Focus 32, Table 1 for the probability of having at least one 
# car in the household by age and education level
# source: https://ibsa.brussels/sites/default/files/publication/documents/Focus-32_FR_rb%20%281%29.pdf

In [ ]:
import pandas as pd

# Avg number of cars per working agent per statistical sector

In [ ]:
# Load population and cars per statistical sector
pop = pd.read_csv('../3_work_assignment/output/workers_with_work_addresses.csv')
sector_stats = pd.read_csv('input_data/cars_per_hh_ss_2021_cleaned.csv')

cars_per_sector = sector_stats.rename(
    columns={
        'CD_SECTOR': 'sector_id',
        'NUM_HOUSEHOLDS': 'n_households',
        'NUM_CARS': 'n_cars',
    }
)

In [ ]:
# Load and calculate the number of people over 20 per statistical sector
age5_sector = pd.read_csv("input_data/TF_CENSUS_2021_S01_cleaned.csv")
age5_sector = age5_sector.rename(columns={"CD_SECTOR": "sector_id"})
age5_sector = age5_sector.rename(columns={"CD_AGE": "age_group"})
age5_sector = age5_sector.rename(columns={"MS_POP": "population"})

# Filter to only get the population over 20 and sum by statistical sector
age_groups_over_20 = [
    "20-24", "25-29", "30-34", "35-39", "40-44",
    "45-49", "50-54", "55-59", "60-64", "65-69",
    "70-74", "75-79", "80-84", "85-89", "90-94",
    "95-99", "100+"
]

age5_sector = age5_sector[age5_sector["age_group"].isin(age_groups_over_20)].copy()

num_adults_per_sector = age5_sector.groupby("sector_id")["population"].sum().reset_index()
num_adults_per_sector = num_adults_per_sector.rename(columns={"population": "n_adults"})

In [ ]:
# Load and calculate the number of employed people per statistical sector
employed_sector = pd.read_csv("input_data/TF_CENSUS_2021_S12_cleaned.csv")
employed_sector = employed_sector.rename(columns={"CD_SECTOR": "sector_id"})
employed_sector = employed_sector.rename(columns={"CD_EMPLOYMENT": "employment_status"})
employed_sector = employed_sector.rename(columns={"MS_POP": "population"})

# Filter to only get the employed population and sum by statistical sector
employed_sector = employed_sector[employed_sector["employment_status"] == "EMP"].copy()

num_employed_per_sector = employed_sector.groupby("sector_id")["population"].sum().reset_index()
num_employed_per_sector = num_employed_per_sector.rename(columns={"population": "n_employed"})

In [ ]:
def compute_avg_num_cars(num_adults, num_employed, n_cars, employed_factor=3):
    """
    Derive the average number of cars per employed adult in a sector, given that employed adults are more likely to own a car than non-employed adults.
    The employed_factor parameter controls how much more likely it is for an employed adult to own a car compared to a non-employed adult.
    """
    num_unemployed = num_adults - num_employed
    avg_num_cars = n_cars * employed_factor / (num_employed * employed_factor + num_unemployed)

    return avg_num_cars

In [ ]:
# Merge the datasets to get the number of adults, employed people, and cars per sector
sector_probs = pd.merge(num_adults_per_sector, num_employed_per_sector, on="sector_id")
sector_probs = pd.merge(sector_probs, cars_per_sector, on="sector_id")

In [ ]:
# Get the car ownership probability for each sector
sector_probs["avg_num_cars"] = sector_probs.apply(
    lambda row: compute_avg_num_cars(
        row["n_adults"],
        row["n_employed"],
        row["n_cars"]
    ),
    axis=1      
)

sector_probs["num_cars_employed"] = sector_probs["avg_num_cars"] * sector_probs["n_employed"]
sector_stats = sector_probs.copy()

In [ ]:
print(f'Total number of adults: {sector_probs["n_adults"].sum()}')
print(f'Total number of cars: {sector_probs["n_cars"].sum()}')
print(f'Average number of cars per adult across all sectors all adults: {sector_probs["n_cars"].sum() / sector_probs["n_adults"].sum():.4f}')

print(f'\nTotal number of employed adults: {sector_probs["n_employed"].sum()} ({sector_probs["n_employed"].sum() / sector_probs["n_adults"].sum() * 100:.2f}%)')
print(f'Total number of cars to be assigned to employed adults: {sector_probs["num_cars_employed"].sum():.2f} ({sector_probs["num_cars_employed"].sum() / sector_probs["n_cars"].sum() * 100:.2f}%)')

In [ ]:
print(sector_probs["avg_num_cars"].describe())
print()
print(sector_probs.sort_values("avg_num_cars", ascending=False).head(10))

print(f'\nNumber of sectors with avg_num_cars > 1: {(sector_probs["avg_num_cars"] > 1).sum()}')
print(f'Number of people living in sectors with avg_num_cars > 1: {sector_probs[sector_probs["avg_num_cars"] > 1]["n_adults"].sum()}')
print(f'Percentage of people living in sectors with avg_num_cars > 1: {sector_probs[sector_probs["avg_num_cars"] > 1]["n_adults"].sum() / sector_probs["n_adults"].sum() * 100:.2f}%')

# Sample car ownership for each agent

In [ ]:
def parse_age_bracket(age_str):
    """Extract lower bound from age bracket string like '25-29', '100+'."""
    if pd.isna(age_str):
        return 0
    age_str = str(age_str).strip()
    if '+' in age_str:
        return int(age_str.replace('+', ''))
    return int(age_str.split('-')[0])

In [ ]:
# Focus 32, Table 1
# Probability of having at least one car per household by age group
age_group_weights = {
    (15, 29): 0.36,
    (30, 34): 0.55,
    (35, 44): 0.58,
    (45, 54): 0.59,
    (55, 64): 0.53,
    (65, 199): 0.62,
}

# Probability of having at least one car per household by educational attainment
education_weights = {
    'primary': 0.36,
    'secondary': 0.48,
    'university': 0.59,
    'unknown': 0.55,
}

In [ ]:
pop['age_bracket'] = pop['age'].apply(parse_age_bracket)

# Education mapping
ISCED11_TO_FOCUS32 = {
    'ISCED11_1': 'primary',
    'ISCED11_2': 'secondary',
    'ISCED11_3': 'secondary',
    'ISCED11_4': 'secondary',
    'ISCED11_5': 'university',
    'ISCED11_6': 'university',
    'ISCED11_7': 'university',
    'ISCED11_8': 'university',
    'UNK': 'unknown',
    'NONE': 'unknown',
}

# Apply to your population
pop['education_category'] = pop['education'].map(ISCED11_TO_FOCUS32).fillna('unknown')

In [ ]:
def get_age_adjustment(age_lower):
    for (lo, hi), adj in age_group_weights.items():
        if lo <= age_lower <= hi:
            return adj
    return 0.0

pop['w_age'] = pop['age_bracket'].apply(get_age_adjustment).fillna(0.1)
pop['w_edu'] = pop['education_category'].map(education_weights).fillna(0.1)

pop['score'] = (pop['w_age'] * pop['w_edu'])

In [ ]:
def assign_cars_to_sector(sector_group):
    sector_id = sector_group.name
    
    sector_row = sector_stats.loc[sector_stats['sector_id'] == sector_id]
    if sector_row.empty:
        return sector_group

    # How many cars does this specific sector get? (Default to 0 if not found)
    total_cars = sector_row['num_cars_employed'].iat[0]
    num_employees = len(sector_group)
    cars_needed = int(min(total_cars, num_employees))
    
    if cars_needed > 0:
        # Sample agents without replacement using their composite score as the probability weight
        winners = sector_group.sample(
            n=cars_needed, 
            replace=False, 
            weights='score', 
            random_state=42
        )
        
        # Flag winners
        col_idx = sector_group.columns.get_loc('has_car')
        sector_group.loc[winners.index, 'has_car'] = True
    
    return sector_group

In [ ]:
# Initialize the 'has_car' column to False for all agents
pop['has_car'] = False

assigned_employed_df = pop.groupby('assigned_sector', group_keys=False).apply(assign_cars_to_sector)

pop.update(assigned_employed_df[['has_car']])

In [ ]:
# Verify the totals
total_cars_assigned = pop['has_car'].sum()
print(f'Total cars assigned: {total_cars_assigned}')

In [ ]:
# Save results
pop.to_csv('output/workers_with_cars.csv', index=False)

### Quick check

In [ ]:
# Which sectors are in the population but missing from census?
pop_sectors    = set(pop['assigned_sector'].unique())
census_sectors = set(sector_probs['sector_id'].unique())

missing = pop_sectors - census_sectors
print(f"Sectors in population but missing from census: {len(missing)}")
print(sorted(missing)[:20])

# How many agents are affected?
n_affected = pop[pop['assigned_sector'].isin(missing)].shape[0]
print(f"Agents affected: {n_affected} ({n_affected/len(pop):.1%})")